In [12]:
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from torch import nn

PROJECT_DIR = Path(r"D:\multimodal_emotion_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MANIFEST_PATH = DATA_DIR / "manifest.csv"
SRC_DIR = PROJECT_DIR / "src"
OUTPUT_DIR = PROJECT_DIR / "results" / "notebook_runs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 13
BATCH_SIZE = 4
NUM_CLASSES = 8
DEVICE = torch.device("cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest[
    manifest["preprocessing_status"] == "ok"
].copy()

print("Valid clips:", len(manifest))
print("Actors:", sorted(manifest["actor_id"].unique()))
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)

Valid clips: 360
Actors: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Device: cpu
Output directory: D:\multimodal_emotion_project\results\notebook_runs


In [13]:
class EmotionDataset(Dataset):
    def __init__(self, dataframe, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment

        self.mean = torch.tensor(
            [0.485, 0.456, 0.406],
            dtype=torch.float32
        ).view(1, 3, 1, 1)

        self.std = torch.tensor(
            [0.229, 0.224, 0.225],
            dtype=torch.float32
        ).view(1, 3, 1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        audio_path = Path(row["audio_feature"])
        video_path = Path(row["video_feature"])

        if not audio_path.is_absolute():
            audio_path = PROJECT_DIR / audio_path

        if not video_path.is_absolute():
            video_path = PROJECT_DIR / video_path

        # Load preprocessed features
        audio = np.load(audio_path).astype(np.float32)
        video = np.load(video_path).astype(np.float32)

        # Audio: (96, 251) -> (1, 96, 251)
        audio = torch.from_numpy(audio).unsqueeze(0)

        # Video: (T, H, W, C) -> (T, C, H, W)
        video = torch.from_numpy(video).permute(0, 3, 1, 2)
        video = video / 255.0

        # Normalize video for the ResNet backbone
        video = (video - self.mean) / self.std

        # Optional audio augmentation for training only
        if self.augment:
            if torch.rand(1).item() < 0.35:
                audio = audio + torch.randn_like(audio) * 0.08

            if torch.rand(1).item() < 0.25:
                audio = audio * (0.5 + torch.rand(1).item())

        # RAVDESS emotion codes are 1 through 8
        label = int(row["emotion_code"]) - 1

        return audio, video, label, str(row["clip_id"])


print("EmotionDataset class created successfully.")

EmotionDataset class created successfully.


In [14]:
actors = sorted(manifest["actor_id"].astype(int).unique())

assert len(actors) == 6, (
    f"Expected 6 actors, but found {len(actors)}: {actors}"
)

# Reproducible actor order
rng = np.random.default_rng(SEED)
shuffled_actors = actors.copy()
rng.shuffle(shuffled_actors)

test_actor = shuffled_actors[0]
val_actor = shuffled_actors[1]
train_actors = shuffled_actors[2:]

train_df = manifest[
    manifest["actor_id"].isin(train_actors)
].copy()

val_df = manifest[
    manifest["actor_id"] == val_actor
].copy()

test_df = manifest[
    manifest["actor_id"] == test_actor
].copy()

print("Training actors:", sorted(train_actors))
print("Validation actor:", val_actor)
print("Test actor:", test_actor)

print("\nTraining clips:", len(train_df))
print("Validation clips:", len(val_df))
print("Test clips:", len(test_df))

assert set(train_actors).isdisjoint({val_actor, test_actor})
assert val_actor != test_actor
assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0

print("\nActor-independent split verified!")

Training actors: [np.int64(2), np.int64(4), np.int64(5), np.int64(6)]
Validation actor: 3
Test actor: 1

Training clips: 240
Validation clips: 60
Test clips: 60

Actor-independent split verified!


In [16]:
train_dataset = EmotionDataset(train_df, augment=True)
val_dataset = EmotionDataset(val_df, augment=False)
test_dataset = EmotionDataset(test_df, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

audio_batch, video_batch, labels_batch, clip_ids = next(iter(train_loader))

print("Audio batch shape:", tuple(audio_batch.shape))
print("Video batch shape:", tuple(video_batch.shape))
print("Labels shape:", tuple(labels_batch.shape))
print("Labels:", labels_batch.tolist())
print("Example clip IDs:", list(clip_ids))

Audio batch shape: (4, 1, 96, 251)
Video batch shape: (4, 8, 3, 224, 224)
Labels shape: (4,)
Labels: [7, 7, 6, 3]
Example clip IDs: ['01-01-08-02-02-01-05', '01-01-08-02-02-02-04', '01-01-07-02-01-02-04', '01-01-04-02-02-02-06']


In [17]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models import AudioCNN

print("train_df exists:", "train_df" in globals())
print("train_dataset exists:", "train_dataset" in globals())
print("AudioCNN imported successfully")

train_df exists: True
train_dataset exists: True
AudioCNN imported successfully


In [19]:
# Diagnostic: Can the CNN overfit a small batch?

debug_model = AudioCNN(num_classes=NUM_CLASSES).to(DEVICE)

samples = [train_dataset[i] for i in range(16)]

audio_batch = torch.stack([s[0] for s in samples]).to(DEVICE)
labels_batch = torch.tensor(
    [s[2] for s in samples],
    dtype=torch.long,
    device=DEVICE
)

debug_criterion = nn.CrossEntropyLoss()
debug_optimizer = torch.optim.Adam(
    debug_model.parameters(),
    lr=1e-3
)

for step in range(100):
    debug_model.train()

    debug_optimizer.zero_grad()
    logits = debug_model(audio_batch)
    loss = debug_criterion(logits, labels_batch)
    loss.backward()
    debug_optimizer.step()

    if (step + 1) % 20 == 0:
        predictions = logits.argmax(dim=1)
        accuracy = (predictions == labels_batch).float().mean().item()

        print(
            f"Step {step + 1}: "
            f"Loss = {loss.item():.4f}, "
            f"Batch accuracy = {accuracy * 100:.1f}%"
        )

Step 20: Loss = 0.7299, Batch accuracy = 75.0%
Step 40: Loss = 0.4840, Batch accuracy = 87.5%
Step 60: Loss = 0.2158, Batch accuracy = 87.5%
Step 80: Loss = 0.1272, Batch accuracy = 93.8%
Step 100: Loss = 0.0324, Batch accuracy = 100.0%


In [21]:
import numpy as np

# Calculate class weights from training data only
class_counts = np.bincount(
    train_df["emotion_code"].astype(int).to_numpy() - 1,
    minlength=NUM_CLASSES
)

class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = class_weights / class_weights.mean()

print("Class weights:")

emotion_mapping = (
    manifest[["emotion_code", "emotion"]]
    .drop_duplicates()
    .sort_values("emotion_code")
)

for (_, row), weight in zip(
    emotion_mapping.iterrows(),
    class_weights
):
    print(f"{row['emotion']}: {weight:.3f}")

Class weights:
neutral: 1.778
calm: 0.889
happy: 0.889
sad: 0.889
angry: 0.889
fearful: 0.889
disgust: 0.889
surprised: 0.889


In [22]:
import sys
import time
import numpy as np
import torch
from pathlib import Path
from torch import nn
from sklearn.metrics import accuracy_score, f1_score

# Import the existing model
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models import AudioCNN

# Create model
model = AudioCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Calculate class weights using training data only
class_counts = np.bincount(
    train_df["emotion_code"].astype(int).to_numpy() - 1,
    minlength=NUM_CLASSES
)

class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = class_weights / class_weights.mean()

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

EPOCHS = 10

history = {
    "train_loss": [],
    "val_loss": [],
    "val_macro_f1": [],
    "val_accuracy": []
}

best_val_f1 = -1.0
best_model_path = OUTPUT_DIR / "audio_cnn_best.pt"

def run_audio_epoch(model, loader, training):
    model.train(training)
    total_loss = 0.0
    all_labels, all_preds = [], []

    for audio, video, labels, clip_ids in loader:
        audio = audio.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(audio)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())

    average_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(
        all_labels, all_preds,
        average="macro", zero_division=0
    )
    accuracy = accuracy_score(all_labels, all_preds)

    return average_loss, macro_f1, accuracy

start_time = time.perf_counter()

for epoch in range(EPOCHS):
    train_loss, train_f1, train_acc = run_audio_epoch(
        model, train_loader, training=True
    )

    val_loss, val_f1, val_acc = run_audio_epoch(
        model, val_loader, training=False
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_macro_f1"].append(val_f1)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_acc:.4f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation accuracy: {val_acc:.4f} | "
        f"Validation macro F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print("  Best model checkpoint saved.")

elapsed = time.perf_counter() - start_time

print(f"\nTraining time: {elapsed:.1f} seconds")
print(f"Best validation macro F1: {best_val_f1:.4f}")
print(f"Best model path: {best_model_path}")

Epoch 1/10 | Train loss: 2.0985 | Train accuracy: 0.1458 | Validation loss: 2.0637 | Validation accuracy: 0.1333 | Validation macro F1: 0.0294
  Best model checkpoint saved.
Epoch 2/10 | Train loss: 2.0734 | Train accuracy: 0.1458 | Validation loss: 2.0345 | Validation accuracy: 0.2500 | Validation macro F1: 0.1509
  Best model checkpoint saved.
Epoch 3/10 | Train loss: 2.0682 | Train accuracy: 0.1583 | Validation loss: 2.0293 | Validation accuracy: 0.2833 | Validation macro F1: 0.1643
  Best model checkpoint saved.
Epoch 4/10 | Train loss: 2.0562 | Train accuracy: 0.1667 | Validation loss: 2.0356 | Validation accuracy: 0.2000 | Validation macro F1: 0.1042
Epoch 5/10 | Train loss: 2.0589 | Train accuracy: 0.1625 | Validation loss: 2.1545 | Validation accuracy: 0.1333 | Validation macro F1: 0.0299
Epoch 6/10 | Train loss: 2.0453 | Train accuracy: 0.1833 | Validation loss: 2.0308 | Validation accuracy: 0.2333 | Validation macro F1: 0.1564
Epoch 7/10 | Train loss: 2.0523 | Train accuracy:

In [13]:
from collections import Counter
import torch
import numpy as np

print("Emotion code mapping:")
print(
    manifest[["emotion_code", "emotion"]]
    .drop_duplicates()
    .sort_values("emotion_code")
    .to_string(index=False)
)

print("\nTraining labels:")
print(
    train_df["emotion"].value_counts().sort_index()
)

print("\nValidation labels:")
print(
    val_df["emotion"].value_counts().sort_index()
)

model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for audio, video, labels, clip_ids in val_loader:
        logits = model(audio.to(DEVICE))
        predictions = logits.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(predictions.cpu().numpy())

print("\nActual validation label counts:")
print(Counter(all_labels))

print("\nPredicted validation label counts:")
print(Counter(all_preds))

Emotion code mapping:
 emotion_code   emotion
            1   neutral
            2      calm
            3     happy
            4       sad
            5     angry
            6   fearful
            7   disgust
            8 surprised

Training labels:
emotion
angry        32
calm         32
disgust      32
fearful      32
happy        32
neutral      16
sad          32
surprised    32
Name: count, dtype: int64

Validation labels:
emotion
angry        8
calm         8
disgust      8
fearful      8
happy        8
neutral      4
sad          8
surprised    8
Name: count, dtype: int64

Actual validation label counts:
Counter({np.int64(1): 8, np.int64(2): 8, np.int64(3): 8, np.int64(4): 8, np.int64(5): 8, np.int64(6): 8, np.int64(7): 8, np.int64(0): 4})

Predicted validation label counts:
Counter({np.int64(1): 60})


In [14]:
# Check predictions on training data

model.eval()

train_preds = []
train_actual = []

with torch.no_grad():
    for audio, video, labels, clip_ids in train_loader:
        audio = audio.to(DEVICE)

        outputs = model(audio)
        predictions = outputs.argmax(dim=1)

        train_preds.extend(predictions.cpu().numpy())
        train_actual.extend(labels.numpy())

from collections import Counter
from sklearn.metrics import accuracy_score

print("Training predictions:")
print(Counter(train_preds))

print("\nTraining accuracy:")
print(accuracy_score(train_actual, train_preds))

Training predictions:
Counter({np.int64(1): 206, np.int64(2): 33, np.int64(5): 1})

Training accuracy:
0.1375


In [15]:
model.eval()

all_logits = []
all_labels = []

with torch.no_grad():
    for audio, video, labels, clip_ids in train_loader:
        logits = model(audio.to(DEVICE))
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

print("Logit mean:", all_logits.mean().item())
print("Logit standard deviation:", all_logits.std().item())

probs = torch.softmax(all_logits, dim=1)

print("Mean maximum probability:",
      probs.max(dim=1).values.mean().item())

print("Predicted classes:",
      torch.bincount(all_logits.argmax(dim=1), minlength=8).tolist())

Logit mean: -0.028694001957774162
Logit standard deviation: 0.2716480791568756
Mean maximum probability: 0.18248885869979858
Predicted classes: [0, 211, 29, 0, 0, 0, 0, 0]


In [16]:
row = train_df.iloc[0]
audio = np.load(PROJECT_DIR / row["audio_feature"])

print("Feature shape:", audio.shape)
print("Minimum:", audio.min())
print("Maximum:", audio.max())
print("Mean:", audio.mean())
print("Standard deviation:", audio.std())
print("Contains NaN:", np.isnan(audio).any())
print("Contains infinity:", np.isinf(audio).any())

Feature shape: (96, 251)
Minimum: -2.0
Maximum: 0.0
Mean: -1.642419
Standard deviation: 0.4717137
Contains NaN: False
Contains infinity: False
